# 06 · Pre-registration 2: contrarian label signal on 20 new large caps (Colab T4)

Hypothesis and test are fixed in `PREREGISTRATION_2.md`. **Do not change code or configs between steps.**
Run all scoring first, the reports last. Secrets (Notebook access on): `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, `HF_TOKEN`.
After a disconnect: rerun **Setup**, then the step you were on.

## Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/PREREGISTRATION_2.md'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

%pip install -q ccxt yfinance bitsandbytes accelerate
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)
for k in ('ALPACA_API_KEY', 'ALPACA_SECRET_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)

T = ['googl', 'meta', 'xom', 'unh', 'v', 'hd', 'pg', 'ko', 'lly', 'avgo',
     'cost', 'wmt', 'bac', 'cvx', 'mrk', 'pep', 'adbe', 'crm', 'nflx', 'orcl']
C2 = ' '.join(f'configs/confirm2/{t}.yaml' for t in T)
!nvidia-smi -L
print(ROOT, CODE, len(T), 'assets')

## Step 1 · Prices + news for 20 assets (CPU, ~10 min, no LLM)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --dry-run --root $ROOT --config $C2 2>&1 | grep -E '^\[|Error|error'

## Step 2 · Score real + shuffled (GPU, ~1.2 h; resumable)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant real shuffled --root $ROOT --config $C2

## Step 3 · Reports (CPU, ~2 min) — run once, after Step 2

In [ ]:
# PRIMARY: 5-day horizon (decision) + shuffled placebo
!cd $CODE && python scripts/run_confirm.py --root $ROOT --equities $C2 --out confirm2_5d
# SECONDARY: same scores, next-day return
!cd $CODE && python scripts/run_confirm.py --root $ROOT --equities $C2 --eval-horizon 1 --out confirm2_1d
# SECONDARY: volatility partialled out
!cd $CODE && python scripts/run_artifact.py --root $ROOT --config $C2 --modes labels